# 유사한 단어 찾기 게임

1. 사전 학습된 모델 또는 적절한 데이터셋을 찾는다.
2. 워드 임베딩 모델을 학습시킨다.
3. 단어 유사도가 0.8 이상인 A, B를 랜덤 추출한다.
4. A, B와 대응되는 C를 추출한다.
5. D를 입력 받는다.

=>
A:B = C:D 관계에 대응하는 D를 찾는 게임을 만든다.
ex) A: 산, B: 바다, C: 나무, D: 물

**<출력 예시>**

관계 [ 수긍 : 추락 = 대사관 : ? ]<br>
모델이 예측한 가장 적합한 단어: 잠입<br>
당신의 답변과 모델 예측의 유사도: 0.34<br>
아쉽네요. 더 생각해보세요.

In [1]:
#!pip install fsspec

In [2]:
#!pip install huggingface_hub

In [3]:
#!pip install lxml

In [4]:
import pandas as pd

splits = {'train': 'dp/train-00000-of-00001.parquet', 'validation': 'dp/validation-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/klue/klue/" + splits["train"])

In [5]:
df = df['sentence']

In [6]:
from lxml import etree
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

In [7]:
from konlpy.tag import Okt
from tqdm import tqdm
import re

okt = Okt()

# 기존 불용어 + 확장
ko_stopwords = [
    "은","는","이","가","을","를","과","와","들","도","부터","까지","에","나","너","그","걔","얘",
    "다","하다","되다","같다","있다",
    # 의미 없는 명사성 단어 추가
    "대해","위해","통해","정도","부분","경우","사실","때문","이번","이번에","관련"
]

preprocessed_data = []

for sentence in tqdm(df):
    sentence = re.sub(r"[a-zA-Z]", " ", sentence)   # 영문 제거
    sentence = re.sub(r"\d+", " ", sentence)        # 숫자 제거
    sentence = re.sub(r"[^가-힣\s]", " ", sentence) # 특수문자 제거

    # 품사 태깅
    morphs = okt.pos(sentence, stem=True)

    # 명사만 추출 + 불용어 제거 + 길이 2 이상
    tokens = [
        word for word, tag in morphs
        if tag in ["Noun", "ProperNoun"]
        and word not in ko_stopwords
        and len(word) > 1
    ]

    preprocessed_data.append(tokens)


100%|██████████| 10000/10000 [00:17<00:00, 561.04it/s]


In [8]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=preprocessed_data,
    vector_size=200,  
    sg=1,             
    window=5,
    min_count=3,      
    workers=4,
    epochs=15       
)

In [9]:
import pandas as pd

pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key).head(10)

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
숙소,0.088443,-0.147693,-0.057438,0.315627,0.075782,-0.138832,0.023598,0.684349,-0.635992,0.282665,...,0.043882,-0.174016,0.065171,-0.319013,0.229910,0.257103,0.162814,-0.257925,-0.122469,-0.313379
위치,0.001873,-0.011127,-0.277350,0.447171,0.051913,-0.176984,0.091219,0.559592,-0.491434,0.551831,...,0.045182,-0.262506,-0.072109,-0.457214,0.275794,0.286284,0.178344,-0.159284,-0.093496,-0.222702
호스트,-0.099795,-0.104056,-0.245436,0.401739,0.079845,0.146017,0.032210,0.672774,-0.589414,0.425148,...,0.147261,0.036354,-0.145980,-0.393373,0.463537,0.235195,-0.023633,-0.482453,-0.024222,-0.356409
지난,0.174103,-0.044127,0.010842,-0.194301,0.192812,-0.071678,0.047439,0.043654,-0.250709,-0.059174,...,-0.009035,0.034995,-0.263434,0.166033,0.191335,0.025840,-0.276739,0.007278,0.053284,0.129287
정말,-0.086919,-0.113181,-0.061654,0.365770,0.032199,0.018839,0.110668,0.631277,-0.452872,0.305255,...,0.083040,-0.172636,-0.008114,-0.337994,0.400659,0.210213,0.000487,-0.180563,-0.130155,-0.212695
시간,0.139455,-0.029405,0.162766,-0.205950,0.000402,-0.365706,-0.049677,0.276588,-0.270157,0.097591,...,0.038134,-0.092260,-0.253023,0.012281,0.552535,0.084079,-0.273738,-0.258319,-0.202316,-0.099813
여행,0.258290,-0.105478,0.103451,0.131673,0.443506,0.102006,0.100770,0.733497,-0.238211,0.122476,...,-0.110711,-0.142719,-0.033502,-0.324535,0.086191,0.229179,0.081534,-0.352133,0.109849,-0.036891
사진,0.074572,0.145720,-0.027513,-0.046321,0.039539,-0.267326,0.308132,0.267822,-0.270021,0.109585,...,0.015125,0.061838,-0.285706,-0.217705,0.054097,0.294958,0.161443,0.173999,-0.019812,0.018243
매우,-0.001607,-0.256478,0.021680,0.265269,0.024187,-0.055772,0.058432,0.449347,-0.189278,0.371925,...,0.060488,-0.169669,-0.023364,-0.376112,0.261092,0.184080,-0.103514,-0.186093,-0.083242,-0.233630
한국,0.180019,-0.117816,-0.214708,0.099456,0.531806,0.203775,-0.170553,0.392798,-0.416115,0.022661,...,-0.135270,0.027995,-0.171396,0.206172,0.072016,0.123257,-0.124590,0.123748,0.100150,0.378348


In [10]:
# model : Word2Vec
model.wv.most_similar('남자')

[('무대', 0.8884242177009583),
 ('응원', 0.8824790716171265),
 ('여자', 0.8821690678596497),
 ('주인공', 0.8657299876213074),
 ('스타', 0.8596224784851074),
 ('제작', 0.8581200242042542),
 ('댄스', 0.8559260964393616),
 ('인스타그램', 0.8515722751617432),
 ('화제', 0.850767970085144),
 ('업로드', 0.8482779860496521)]

In [11]:
import random

kv = model.wv

def play():
    vocab = list(kv.key_to_index.keys())

    # A, B 두 단어 랜덤 선택한다.
    A, B = random.sample(vocab, 2)

    # A와 가장 유사한 단어 C 선택한다.
    try:
        C = kv.most_similar(A, topn=1)[0][0]
    except KeyError:
        print("해당 단어로는 유사도 계산 불가. 다시 실행하세요.")
        return

    # 모델 예측: A:B = C:?
    try:
        pred = kv.most_similar(positive=[B, C], negative=[A], topn=1)[0][0]
    except KeyError:
        print("관계 계산 불가. 다시 실행하세요.")
        return

    print(f"관계 [ {A} : {B} = {C} : ? ]")
    print(f"모델이 예측한 가장 적합한 단어: {pred}")

    user = input("D를 입력하세요: ").strip()
    print(f"당신이 선택한 단어: {user}")

    if user in kv and pred in kv:
        sim = kv.similarity(user, pred)
        print(f"당신의 답변과 모델 예측의 유사도: {sim:.2f}")
        if sim < 0.7:
            print("아쉽네요. 더 생각해보세요.")
    else:
        print("사전에 없는 단어라 유사도 계산 불가")

In [12]:
play()

관계 [ 추위 : 구비 = 골대 : ? ]
모델이 예측한 가장 적합한 단어: 토스트
당신이 선택한 단어: 영어
당신의 답변과 모델 예측의 유사도: 0.71
